# 02 — EDA: Singapore
**Data source:** `data/processed/singapore_enriched.parquet`  
**Sections:**
1. Load & quick overview
2. Price distribution by room type (boxplot)
3. Room type breakdown (bar chart)
4. Availability vs occupancy by room type (probability of being used)
5. Neighbourhood — median price ranking
6. Neighbourhood — listing count vs review activity
7. Host portfolio segmentation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

DATA = Path('../data/processed')
df   = pd.read_parquet(DATA / 'singapore_enriched.parquet')

print(f'Rows: {len(df):,}  |  Cols: {df.shape[1]}')
df.head(3)

---
## 1. Price Distribution by Room Type — Boxplot
**Why boxplot:** Shows median, spread (IQR), and outliers simultaneously.  
A bar chart of averages would hide the extreme skew we already know exists.

In [ ]:
# Cap at 99th percentile so outliers don't crush the visible range
p99 = df['price'].quantile(0.99)
plot_df = df[df['price'] <= p99].copy()

room_order = (
    plot_df.groupby('room_type')['price']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=plot_df, x='room_type', y='price',
    order=room_order, palette='Blues_d', ax=ax
)
ax.set_title('Singapore — Price Distribution by Room Type (capped at 99th pct)', fontsize=13, pad=12)
ax.set_xlabel('Room Type')
ax.set_ylabel('Price (SGD / night)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()

print('\nMedian price by room type:')
print(plot_df.groupby('room_type')['price'].median().sort_values(ascending=False).to_string())

---
## 2. Room Type Breakdown — Count Bar Chart
**Why this matters:** Market composition drives pricing dynamics.  
A market dominated by private rooms behaves differently from one dominated by entire homes.

In [ ]:
room_counts = df['room_type'].value_counts()
room_pct    = (room_counts / len(df) * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Count
room_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('Blues_d', len(room_counts)), edgecolor='white')
axes[0].set_title('Listing Count by Room Type', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('Number of Listings')
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(room_counts):
    axes[0].text(i, v + 10, str(v), ha='center', fontsize=9)

# Percentage pie
axes[1].pie(
    room_counts, labels=room_counts.index,
    autopct='%1.1f%%', startangle=90,
    colors=sns.color_palette('Blues_d', len(room_counts))
)
axes[1].set_title('Market Share by Room Type', fontsize=12)

plt.suptitle('Singapore — Room Type Composition', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Availability vs Occupancy by Room Type
**Probability of being used:**  
`availability_365` = days the listing is open for booking.  
Low availability → host has blocked most days → listing is likely frequently booked.  
`occupancy_proxy = (365 - availability_365) / 365` — our estimated occupancy rate.

**Why compare room types here:** Different room types may have structurally different booking patterns — entire homes may be blocked out for owner personal use, not just bookings.

In [ ]:
avail_stats = (
    df.groupby('room_type')
    .agg(
        median_availability = ('availability_365', 'median'),
        median_occupancy_pct = ('occupancy_proxy', lambda x: round(x.median() * 100, 1)),
        listing_count = ('id', 'count')
    )
    .sort_values('median_occupancy_pct', ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Median availability (lower = more booked)
colors = sns.color_palette('Blues_d', len(avail_stats))
bars = axes[0].bar(avail_stats['room_type'], avail_stats['median_availability'], color=colors, edgecolor='white')
axes[0].set_title('Median Days Available per Year\n(lower = more likely in use)', fontsize=11)
axes[0].set_ylabel('Days Available (out of 365)')
axes[0].set_ylim(0, 365)
axes[0].axhline(y=365, color='red', linestyle='--', alpha=0.4, label='max (never booked)')
axes[0].legend(fontsize=8)
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, avail_stats['median_availability']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4, f'{val:.0f}d', ha='center', fontsize=9)

# Occupancy proxy %
bars2 = axes[1].bar(avail_stats['room_type'], avail_stats['median_occupancy_pct'], color=colors, edgecolor='white')
axes[1].set_title('Estimated Occupancy Rate by Room Type\n(proxy: 1 - availability/365)', fontsize=11)
axes[1].set_ylabel('Occupancy Estimate (%)')
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars2, avail_stats['median_occupancy_pct']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val}%', ha='center', fontsize=9)

plt.suptitle('Singapore — Room Type: Availability & Occupancy', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print(avail_stats[['room_type','median_availability','median_occupancy_pct','listing_count']].to_string(index=False))

---
## 4. Neighbourhood — Median Price Ranking
**Why horizontal bar:** Many neighbourhoods — horizontal layout is readable.  
Sorted descending so the most expensive is immediately visible at the top.

In [ ]:
nb_price = (
    df.groupby('neighbourhood')['price']
    .agg(['median', 'count'])
    .rename(columns={'median': 'median_price', 'count': 'listing_count'})
    .query('listing_count >= 5')
    .sort_values('median_price', ascending=False)
    .head(20)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(
    nb_price['neighbourhood'][::-1],
    nb_price['median_price'][::-1],
    color=sns.color_palette('Blues_d', len(nb_price)),
    edgecolor='white'
)
for bar, (_, row) in zip(bars, nb_price[::-1].iterrows()):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'${row["median_price"]:.0f}  (n={row["listing_count"]})',
            va='center', fontsize=8)

ax.set_title('Singapore — Top 20 Neighbourhoods by Median Nightly Price\n(min 5 listings)', fontsize=12, pad=12)
ax.set_xlabel('Median Price (SGD / night)')
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.set_xlim(0, nb_price['median_price'].max() * 1.25)
plt.tight_layout()
plt.show()

---
## 5. Neighbourhood — Listing Count vs Review Activity
**Why scatter plot:** Shows two dimensions at once — volume (how many listings) vs demand (how active are they).  
Quadrant analysis: top-right = high supply + high demand (thriving). Top-left = high demand but low supply (opportunity).

In [ ]:
nb_activity = (
    df.groupby('neighbourhood')
    .agg(
        listing_count       = ('id', 'count'),
        median_reviews_pm   = ('reviews_per_month', 'median'),
        median_price        = ('price', 'median')
    )
    .query('listing_count >= 5')
    .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 7))
scatter = ax.scatter(
    nb_activity['listing_count'],
    nb_activity['median_reviews_pm'],
    c=nb_activity['median_price'],
    cmap='Blues', s=80, alpha=0.8, edgecolors='grey', linewidths=0.4
)
plt.colorbar(scatter, ax=ax, label='Median Price (SGD)')

# Label top neighbourhoods
top = nb_activity.nlargest(8, 'listing_count')
for _, row in top.iterrows():
    ax.annotate(row['neighbourhood'],
                (row['listing_count'], row['median_reviews_pm']),
                textcoords='offset points', xytext=(6, 2), fontsize=7.5)

# Quadrant lines at medians
ax.axvline(nb_activity['listing_count'].median(), color='gray', linestyle='--', alpha=0.5, linewidth=0.9)
ax.axhline(nb_activity['median_reviews_pm'].median(), color='gray', linestyle='--', alpha=0.5, linewidth=0.9)

ax.set_title('Singapore — Neighbourhood: Listing Supply vs Review Activity\n(colour = median price)', fontsize=12, pad=12)
ax.set_xlabel('Number of Listings')
ax.set_ylabel('Median Reviews per Month')
plt.tight_layout()
plt.show()

---
## 6. Neighbourhood — Median Price vs Review Activity (Investment Signal)
**PRD Question 3:** Which neighbourhoods are underpriced relative to their review scores?  
Low price + high review activity = underpriced. High price + low review activity = overpriced.

In [ ]:
nb_top20 = (
    df.groupby('neighbourhood')
    .agg(
        listing_count     = ('id', 'count'),
        median_price      = ('price', 'median'),
        median_reviews_pm = ('reviews_per_month', 'median')
    )
    .query('listing_count >= 5')
    .sort_values('listing_count', ascending=False)
    .head(20)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Median price per neighbourhood (top 20 by listing count)
nb_sorted_price = nb_top20.sort_values('median_price', ascending=True)
axes[0].barh(nb_sorted_price['neighbourhood'], nb_sorted_price['median_price'],
             color=sns.color_palette('Blues_d', len(nb_sorted_price)), edgecolor='white')
axes[0].set_title('Median Price per Night\n(Top 20 neighbourhoods by listing count)', fontsize=11)
axes[0].set_xlabel('Median Price (SGD)')
axes[0].xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))

# Review activity per neighbourhood
nb_sorted_rev = nb_top20.sort_values('median_reviews_pm', ascending=True)
axes[1].barh(nb_sorted_rev['neighbourhood'], nb_sorted_rev['median_reviews_pm'],
             color=sns.color_palette('Oranges_d', len(nb_sorted_rev)), edgecolor='white')
axes[1].set_title('Median Reviews per Month\n(proxy for booking demand)', fontsize=11)
axes[1].set_xlabel('Median Reviews / Month')

plt.suptitle('Singapore — Neighbourhood Price vs Demand Activity', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

---
## 7. Host Portfolio Segmentation
**PRD Question 2:** What % of hosts control the majority of listings?  
Single-listing owners vs small operators vs commercial operators (6+ listings).

In [ ]:
df['host_tier'] = pd.cut(
    df['calculated_host_listings_count'],
    bins=[0, 1, 5, float('inf')],
    labels=['Single (1)', 'Small (2–5)', 'Commercial (6+)']
)

host_seg = (
    df.groupby('host_tier', observed=True)
    .agg(
        host_count    = ('host_id', 'nunique'),
        listing_count = ('id', 'count'),
        median_price  = ('price', 'median')
    )
    .reset_index()
)
host_seg['pct_listings'] = (host_seg['listing_count'] / host_seg['listing_count'].sum() * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = ['#2196F3', '#64B5F6', '#BBDEFB']

# Host count
axes[0].bar(host_seg['host_tier'], host_seg['host_count'], color=palette, edgecolor='white')
axes[0].set_title('Number of Hosts per Tier', fontsize=11)
axes[0].set_ylabel('Host Count')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(host_seg['host_count']):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=9)

# % of market listings
axes[1].bar(host_seg['host_tier'], host_seg['pct_listings'], color=palette, edgecolor='white')
axes[1].set_title('% of Total Listings Controlled', fontsize=11)
axes[1].set_ylabel('% of Listings')
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis='x', rotation=20)
for i, v in enumerate(host_seg['pct_listings']):
    axes[1].text(i, v + 1, f'{v}%', ha='center', fontsize=9)

# Median price per tier
axes[2].bar(host_seg['host_tier'], host_seg['median_price'], color=palette, edgecolor='white')
axes[2].set_title('Median Price by Host Tier', fontsize=11)
axes[2].set_ylabel('Median Price (SGD)')
axes[2].yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
axes[2].tick_params(axis='x', rotation=20)
for i, v in enumerate(host_seg['median_price']):
    axes[2].text(i, v + 2, f'${v:.0f}', ha='center', fontsize=9)

plt.suptitle('Singapore — Host Portfolio Segmentation', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print(host_seg[['host_tier','host_count','listing_count','pct_listings','median_price']].to_string(index=False))